In [13]:
from pathlib import Path
import json
import re
import shutil

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

In [ ]:
DATASETS_TO_NORMALIZE = ["collected", "public"]

FOLDER_LEVELS_UP_BEFORE_PREPROCESSING = 2
PREPROCESSING_FOLDER_NAME = "data_preprocessing"
RAW_FEATURES_DIR_NAME = "Raw_Features"
NORMALIZED_DIR_NAME = "Normalized"

CLEAR_NORMALIZED_DATA = True

TRAIN_RATIO = 0.8
RANDOM_SEED = 42
STATS_SAMPLE_PER_ARRAY = 5000

NORMALIZED_FEATURE_KEYS = {"amplitude_db", "cwt_amplitude_db", "speed_hz_map"}
PHASE_FEATURE_KEYS = {"phase_sin", "phase_cos"}

PREPROCESSING_BASE_DIR = Path.cwd()
for _ in range(FOLDER_LEVELS_UP_BEFORE_PREPROCESSING):
    PREPROCESSING_BASE_DIR = PREPROCESSING_BASE_DIR.parent

PREPROCESSING_ROOT = PREPROCESSING_BASE_DIR / PREPROCESSING_FOLDER_NAME
print("PREPROCESSING_ROOT:", PREPROCESSING_ROOT)

PREPROCESSING_ROOT: c:\Users\szymo\Desktop\Wibracje\data_preprocessing


In [15]:
def natural_key(path):
    return [int(x) if x.isdigit() else x.lower() for x in re.split(r"(\d+)", Path(path).name)]


def load_npy(path):
    return np.load(path, allow_pickle=True).item()


def save_npy(path, data):
    np.save(path, data, allow_pickle=True)


def feature_key_from_component_name(name):
    if " | " in name:
        return name.split(" | ", 1)[1]
    return name


def should_normalize_component(name):
    return feature_key_from_component_name(name) in NORMALIZED_FEATURE_KEYS


def view_indices(component_names, view):
    indices = []
    for idx, name in enumerate(component_names):
        if view == "non_tacho" and name.startswith("Tachometer |"):
            continue
        indices.append(idx)
    return indices


def make_view_payload(payload, view):
    indices = view_indices(payload["component_names"], view)
    names = [payload["component_names"][i] for i in indices]
    out = {
        "X": payload["X"][indices].astype(np.float32),
        "component_names": names,
        "frequency_hz": payload["frequency_hz"].astype(np.float32),
        "time_s": payload["time_s"].astype(np.float32),
        "labels": payload["labels"],
        "metadata": dict(payload["metadata"]),
        "feature_keys": payload.get("feature_keys", []),
        "label_scaling": payload.get("label_scaling", None),
    }
    out["metadata"]["view"] = view
    out["metadata"]["tacho_included_in_X"] = view == "tacho"
    return out


def split_raw_files(raw_files, train_ratio=TRAIN_RATIO, seed=RANDOM_SEED):
    rows = []
    for path in raw_files:
        payload = load_npy(path)
        labels = payload["labels"]
        rows.append({
            "path": path,
            "source_file": payload["metadata"]["source_file"],
            "angle_label": labels["angle_label"],
            "imbalance_actual_gmm": labels["imbalance_actual_gmm"],
        })
    table = pd.DataFrame(rows)
    if len(table) == 0:
        return [], [], table

    rng = np.random.default_rng(seed)
    table["split_label"] = table["angle_label"].astype(str) + "_" + table["imbalance_actual_gmm"].round(6).astype(str)
    train_paths = []
    test_paths = []

    for _, group in table.groupby("split_label"):
        paths = list(group["path"])
        rng.shuffle(paths)
        if len(paths) == 1:
            train_paths.extend(paths)
            continue
        n_train = int(round(len(paths) * train_ratio))
        n_train = max(1, min(n_train, len(paths) - 1))
        train_paths.extend(paths[:n_train])
        test_paths.extend(paths[n_train:])

    rng.shuffle(train_paths)
    rng.shuffle(test_paths)
    table["split"] = table["path"].apply(lambda p: "train" if p in train_paths else "test")
    return train_paths, test_paths, table

In [16]:
def sample_values_for_stats(files, view, rng):
    sampled = {}
    for path in tqdm(files, desc=f"Sampling stats | {view}"):
        payload = make_view_payload(load_npy(path), view)
        X = payload["X"]
        names = payload["component_names"]
        for idx, name in enumerate(names):
            if not should_normalize_component(name):
                continue
            values = X[idx].ravel()
            values = values[np.isfinite(values)]
            if len(values) == 0:
                continue
            if len(values) > STATS_SAMPLE_PER_ARRAY:
                values = values[rng.choice(len(values), size=STATS_SAMPLE_PER_ARRAY, replace=False)]
            sampled.setdefault(name, []).append(values.astype(np.float32))
    return sampled


def percentile_range(values):
    p5 = float(np.percentile(values, 5))
    p95 = float(np.percentile(values, 95))
    if np.isclose(p5, p95):
        p5 = float(np.min(values))
        p95 = float(np.max(values))
    if np.isclose(p5, p95):
        p5 -= 1.0
        p95 += 1.0
    return p5, p95


def fit_normalization_stats(files, view, seed=RANDOM_SEED):
    rng = np.random.default_rng(seed)
    sampled = sample_values_for_stats(files, view, rng)
    stats = {}
    for name, chunks in sampled.items():
        values = np.concatenate(chunks)
        p5, p95 = percentile_range(values)
        stats[name] = {
            "component_name": name,
            "feature_key": feature_key_from_component_name(name),
            "p5": p5,
            "p95": p95,
            "min_sampled": float(np.min(values)),
            "max_sampled": float(np.max(values)),
            "mean_sampled": float(np.mean(values)),
            "std_sampled": float(np.std(values)),
            "n_sampled": int(len(values)),
        }
    return stats


def normalize_array(arr, component_name, stats):
    if not should_normalize_component(component_name):
        return arr.astype(np.float32)
    if component_name not in stats:
        return arr.astype(np.float32)
    p5 = stats[component_name]["p5"]
    p95 = stats[component_name]["p95"]
    arr = np.clip(arr, p5, p95)
    arr = 2 * (arr - p5) / (p95 - p5) - 1
    return arr.astype(np.float32)


def normalize_payload(payload, stats):
    X = payload["X"].copy().astype(np.float32)
    for idx, name in enumerate(payload["component_names"]):
        X[idx] = normalize_array(X[idx], name, stats)
    out = dict(payload)
    out["X"] = X
    out["normalization"] = {
        "method": "train_percentile_p5_p95_to_minus1_plus1",
        "normalizes": sorted(NORMALIZED_FEATURE_KEYS),
        "phase_copied": sorted(PHASE_FEATURE_KEYS),
    }
    out["metadata"] = dict(payload["metadata"])
    out["metadata"]["normalized"] = True
    return out

In [17]:
def save_split_files(raw_paths, split, view, out_dir, stats):
    out_dir.mkdir(parents=True, exist_ok=True)
    rows = []
    for i, path in enumerate(tqdm(raw_paths, desc=f"Saving {view}/{split}")):
        raw = load_npy(path)
        payload = make_view_payload(raw, view)
        payload = normalize_payload(payload, stats)
        labels = payload["labels"]
        meta = payload["metadata"]
        out_name = f"{Path(path).stem}_{view}_{split}.npy"
        out_path = out_dir / out_name
        save_npy(out_path, payload)
        rows.append({
            "path": str(out_path),
            "file": out_name,
            "dataset": meta["dataset"],
            "view": view,
            "split": split,
            "source_file": meta["source_file"],
            "source_has_tachometer": meta["source_has_tachometer"],
            "angle_label": labels["angle_label"],
            "angle_deg": labels["angle_deg"],
            "imbalance_actual_gmm": labels["imbalance_actual_gmm"],
            "imbalance_relative": labels["imbalance_relative"],
            "X_shape": str(payload["X"].shape),
        })
    return rows


def normalize_dataset(dataset_name):
    dataset_root = PREPROCESSING_ROOT / dataset_name
    raw_dir = dataset_root / RAW_FEATURES_DIR_NAME
    normalized_root = dataset_root / NORMALIZED_DIR_NAME
    if CLEAR_NORMALIZED_DATA and normalized_root.exists():
        shutil.rmtree(normalized_root)
    normalized_root.mkdir(parents=True, exist_ok=True)

    raw_files = sorted(raw_dir.glob("*.npy"), key=natural_key)
    if not raw_files:
        print(f"No raw feature files found for {dataset_name}: {raw_dir}")
        return pd.DataFrame()

    train_raw, test_raw, split_table = split_raw_files(raw_files)
    split_table.to_csv(normalized_root / "source_split.csv", index=False)

    views = ["non_tacho"]
    if any(load_npy(path)["metadata"].get("source_has_tachometer", False) for path in raw_files):
        views.append("tacho")

    all_rows = []
    all_stats = {}

    for view in views:
        train_for_view = []
        test_for_view = []
        for path in train_raw:
            payload = load_npy(path)
            if view == "tacho" and not payload["metadata"].get("source_has_tachometer", False):
                continue
            train_for_view.append(path)
        for path in test_raw:
            payload = load_npy(path)
            if view == "tacho" and not payload["metadata"].get("source_has_tachometer", False):
                continue
            test_for_view.append(path)

        stats = fit_normalization_stats(train_for_view, view=view)
        all_stats[view] = stats

        for split, paths in [("train", train_for_view), ("test", test_for_view)]:
            out_dir = normalized_root / view / split
            all_rows.extend(save_split_files(paths, split, view, out_dir, stats))

    with open(normalized_root / "normalization_stats.json", "w", encoding="utf-8") as f:
        json.dump(all_stats, f, indent=2, ensure_ascii=False)

    manifest = pd.DataFrame(all_rows)
    manifest.to_csv(normalized_root / "manifest.csv", index=False)
    print(f"{dataset_name}: saved {len(manifest)} normalized files to {normalized_root}")
    display(manifest.groupby(["view", "split"]).size())
    return manifest

In [18]:
all_manifests = []
for dataset_name in DATASETS_TO_NORMALIZE:
    manifest = normalize_dataset(dataset_name)
    if len(manifest):
        all_manifests.append(manifest)

if all_manifests:
    combined_manifest = pd.concat(all_manifests, ignore_index=True)
    display(combined_manifest.head())
    display(combined_manifest.groupby(["dataset", "view", "split"]).size())
print("Done.")

Sampling stats | non_tacho:   0%|          | 0/462 [00:00<?, ?it/s]

Saving non_tacho/train:   0%|          | 0/462 [00:00<?, ?it/s]

Saving non_tacho/test:   0%|          | 0/123 [00:00<?, ?it/s]

Sampling stats | tacho:   0%|          | 0/308 [00:00<?, ?it/s]

Saving tacho/train:   0%|          | 0/308 [00:00<?, ?it/s]

...

public: saved 382 normalized files to c:\Users\szymo\Desktop\Wibracje\data_preprocessing\public\Normalized


view       split
non_tacho  test      77
           train    305
dtype: int64

,path,file,dataset,view,split,source_file,source_has_tachometer,angle_label,angle_deg,imbalance_actual_gmm,imbalance_relative,X_shape
0,c:\Users\szymo\Desktop\Wibracje\data_preproces...,v04_14_a11_zew3_processed_features_non_tacho_t...,collected,non_tacho,train,v04_14_a11_zew3_processed.csv,True,11,232.941176,337.0,0.717021,"(9, 500, 41)"
1,c:\Users\szymo\Desktop\Wibracje\data_preproces...,v10_14_a5_sr_processed_features_non_tacho_trai...,collected,non_tacho,train,v10_14_a5_sr_processed.csv,False,5,105.882353,161.1,0.342766,"(9, 500, 41)"
2,c:\Users\szymo\Desktop\Wibracje\data_preproces...,v13_14_a11_zew_sr_processed_features_non_tacho...,collected,non_tacho,train,v13_14_a11_zew_sr_processed.csv,True,11,232.941176,336.5,0.715957,"(9, 500, 41)"
3,c:\Users\szymo\Desktop\Wibracje\data_preproces...,v05_14_a5_zew2_processed_features_non_tacho_tr...,collected,non_tacho,train,v05_14_a5_zew2_processed.csv,False,5,105.882353,261.0,0.555319,"(9, 500, 41)"
4,c:\Users\szymo\Desktop\Wibracje\data_preproces...,v10_14_a1_sr_processed_features_non_tacho_trai...,collected,non_tacho,train,v10_14_a1_sr_processed.csv,False,1,21.176471,161.1,0.342766,"(9, 500, 41)"


dataset    view       split
collected  non_tacho  test     123
                      train    462
           tacho      test      81
                      train    308
public     non_tacho  test      77
                      train    305
dtype: int64

Done.
